In [23]:
# BIDMC RESEARCHER INPUT SPECIFICATION
# This is the complete human-authored scientific input. The execution layer may
# implement repetition, matching, validation, aggregation, and reporting, but it
# must not introduce a scientific rule that is absent from this cell.

import featuregraph as fg
import pandas as pd

# Study scope
subject_id = 1                         # development and inspection subject
subject_ids = list(range(1, 54))       # frozen cohort
sampling_rate_hz = 125
signal_column = 'respiration'
expected_samples_per_subject = 60001

# Investigator-selected construction parameters
smooth_window = 100
numerical_atol = 1e-12                 # numerical, not physiological
effective_support_samples = 2 * smooth_window - 1
effective_support_seconds = effective_support_samples / sampling_rate_hz
construction_is_causal = False         # negative shift uses future samples

# One-subject observation table. The execution layer applies this same specification
# independently to every subject_id; stateful operations may never cross subjects.
df = fg.datasets.bidmc(subject=subject_id).copy()
df['subject_id'] = subject_id
df['sample_index'] = df.index
df['time_seconds'] = df['sample_index'] / sampling_rate_hz

# Preserve the raw signal and construct a separate offline envelope.
df['respiration_smooth'] = (
    df['respiration']
    .rolling(smooth_window, min_periods=smooth_window)
    .max()
    .rolling(smooth_window, min_periods=smooth_window)
    .mean()
    .shift(-smooth_window)
)

# Primitive states
df['respiration_change'] = df['respiration_smooth'].diff()
df['respiration_smooth_valid'] = (
    df['respiration_smooth'].notna()
    & df['respiration_change'].notna()
)
valid = df['respiration_smooth_valid']
df['respiration_rising'] = valid & df['respiration_change'].gt(numerical_atol)
df['respiration_falling'] = valid & df['respiration_change'].lt(-numerical_atol)
df['respiration_inactive'] = valid & df['respiration_change'].abs().le(numerical_atol)

# Boundary events
rising_integer = df['respiration_rising'].astype(int)
df['enter_respiration_rising'] = valid & rising_integer.diff().eq(1)
df['exit_respiration_rising'] = valid & rising_integer.diff().eq(-1)

# Directional states describe the edge ending at the current row, so the extremum
# itself is assigned to the preceding sample. Entering rising marks a trough; exiting
# rising marks a peak. Successive troughs bound one candidate wave.
df['trough_event'] = df['enter_respiration_rising'].shift(-1, fill_value=False)
df['peak_event'] = df['exit_respiration_rising'].shift(-1, fill_value=False)
df['wave_id'] = df['enter_respiration_rising'].cumsum()

# Flat extrema are intervals, not arbitrary single samples. Adjacent envelope values
# whose absolute difference is <= numerical_atol belong to the same flat run. Each
# boundary is projected to the integer midpoint of its complete flat run.
plateau_projection = {
    'same_run': 'abs(current_value - preceding_value) <= numerical_atol',
    'representative_index': 'start_index + (end_index - start_index) // 2',
}

# One object is a trough-peak-trough interval. Incomplete and ambiguous candidates
# must be retained with flags rather than silently treated as complete or discarded.
object_definition = {
    'name': 'respiratory_wave',
    'identity': 'wave_id created by cumulative entering-rising events',
    'start': 'midpoint of starting trough interval',
    'peak': 'midpoint of the peak interval',
    'end': 'midpoint of the following trough interval',
    'complete_requires': [
        'start trough exists',
        'peak exists',
        'following end trough exists',
        'start_index < peak_index < end_index',
        'candidate is not a leading or terminal boundary fragment',
        'projected plateau intervals remain strictly ordered',
    ],
}

# Required object-table schema and exact property meanings
object_properties = {
    'subject_id': 'BIDMC subject identifier',
    'wave_id': 'within-subject identifier',
    'start_index': 'starting-trough midpoint',
    'peak_index': 'peak midpoint',
    'end_index': 'ending-trough midpoint',
    'start_time_seconds': 'start_index / sampling_rate_hz',
    'peak_time_seconds': 'peak_index / sampling_rate_hz',
    'end_time_seconds': 'end_index / sampling_rate_hz',
    'rise_duration_seconds': '(peak_index - start_index) / sampling_rate_hz',
    'fall_duration_seconds': '(end_index - peak_index) / sampling_rate_hz',
    'duration_seconds': '(end_index - start_index) / sampling_rate_hz',
    'period_seconds': '(peak_index - preceding_complete_peak_index) / sampling_rate_hz',
    'rate_bpm': '60 / period_seconds',
    'raw_minimum': 'minimum raw respiration from start_index through end_index',
    'raw_maximum': 'maximum raw respiration from start_index through end_index',
    'full_excursion': 'raw_maximum - raw_minimum',
    'temporal_symmetry': (
        '1 - abs(rise_duration_seconds - fall_duration_seconds) / duration_seconds'
    ),
    'start_trough_start_index': 'first sample in starting trough plateau',
    'start_trough_end_index': 'last sample in starting trough plateau',
    'peak_start_index': 'first sample in peak plateau',
    'peak_end_index': 'last sample in peak plateau',
    'end_trough_start_index': 'first sample in ending trough plateau',
    'end_trough_end_index': 'last sample in ending trough plateau',
    'is_complete': 'all complete-object requirements are satisfied',
    'plateau_boundary_ambiguous': 'projected extremum intervals overlap or are unordered',
    'boundary_truncated': 'candidate lacks a complete leading or trailing boundary',
}

# Frozen external comparator. It does not define FeatureGraph objects.
comparator = {
    'input': 'raw respiration',
    'filter': 'fourth-order Butterworth low-pass',
    'cutoff_hz': 0.8,
    'application': 'scipy.signal.sosfiltfilt',
    'peak_function': 'scipy.signal.find_peaks',
    'trough_function': 'scipy.signal.find_peaks applied to the negative filtered signal',
    'minimum_distance_samples': 188,
    'minimum_prominence': 0.08,
    'complete_object': 'exactly one peak strictly between successive troughs',
}

# Ordered one-to-one comparison
matching = {
    'anchor': 'peak_index',
    'tolerance_samples': 63,
    'ordered': True,
    'one_to_one': True,
    'objective': [
        'maximize matched-object count',
        'then minimize total absolute peak-index error',
    ],
    'classes': ['matched', 'FeatureGraph-only', 'comparator-only'],
}

# Independent BIDMC breath annotations
annotation_comparison = {
    'columns': [
        'breaths ann1 [signal sample no]',
        'breaths ann2 [signal sample no]',
    ],
    'anchor': 'FeatureGraph peak midpoint',
    'tolerance_samples': 63,
    'FeatureGraph_only_labels': [
        'retained by both annotators',
        'retained by annotator 1 only',
        'retained by annotator 2 only',
        'excluded by both annotators',
    ],
    'constraint': 'annotation-discordant does not mean false or clinically abnormal',
}

# Required mechanical and representational checks
validation_requirements = [
    'all 53 subjects load successfully with complete files and expected schema',
    'sample indices are ordered and stateful operations never cross subjects',
    'raw respiration remains unchanged',
    'every valid sample has exactly one of rising, falling, or inactive',
    'invalid envelope-edge samples generate no transition events',
    'complete objects satisfy start_index < peak_index < end_index',
    'flat-run projection uses numerical_atol rather than exact float equality',
    'subject 13 residue near samples 33400:33600 creates no numerical chatter',
    'incomplete and ambiguous candidates remain explicitly labeled',
    'cohort totals are protected by regression assertions',
    'software versions and a hash of this specification are recorded',
]

# Requested results
requested_outputs = {
    'tables': [
        'observation states and events',
        'complete FeatureGraph objects',
        'incomplete and plateau-ambiguous candidates',
        'SciPy comparator objects',
        'matched, FeatureGraph-only, and comparator-only objects',
    ],
    'cohort_counts': [
        'detected peaks', 'complete objects', 'matched objects',
        'FeatureGraph-only objects', 'comparator-only objects',
        'ambiguous objects', 'invalidated complete candidates',
    ],
    'agreement_metrics': [
        'peak-index error', 'rate delta', 'period delta',
        'full-excursion delta', 'temporal-symmetry delta',
        'agreement with each BIDMC annotator',
    ],
    'discordance_metrics': [
        'concentration by subject and time', 'annotation-support label',
        'period', 'duration', 'rise duration', 'fall duration',
        'full excursion', 'temporal symmetry',
    ],
    'window_sensitivity': {
        'subject_ids': [1],
        'smooth_windows_samples': [75, 100, 125],
    },
}

# Interpretive boundary
supported_claims = [
    'The pandas specification deterministically constructs bounded wave objects.',
    'The frozen construction executes across all 53 BIDMC records.',
    'The representation exposes states, events, boundaries, properties, and discordance.',
    'FeatureGraph and the SciPy comparator agree on many objects but are not equivalent.',
    'Discordant objects can be localized, measured, labeled, and passed onward.',
]
unsupported_claims = [
    'FeatureGraph is clinically validated or superior to scipy.signal.find_peaks.',
    'FeatureGraph-only objects are necessarily true breaths or false detections.',
    'Annotation-discordant objects represent a particular pathology.',
    'The smoothing window is universally appropriate outside this study.',
]

# Human-LLM boundary
execution_contract = {
    'allowed': (
        'Implement downloading, integrity checks, cohort repetition, object assembly, '
        'matching, validation, aggregation, regression tests, and reporting.'
    ),
    'must_ask_before': (
        'Adding or changing any threshold, filter, boundary, identity, completeness, '
        'matching, exclusion, imputation, property, or scientific interpretation.'
    ),
    'preserve': [
        'this researcher input notebook',
        'complete generated execution code',
        'object-level outputs',
        'validation and integrity report',
        'software environment and dependency versions',
        'results summary',
    ],
}
